# Week 14 — QuakeMigrate quality filtering and final catalogue assembly
Applies the same quality filters used for EQTransformer and PhaseNet
(horizontal error <5km, azimuthal gap <180 deg, min 8 phases, mean RMS
residual <0.5s), for a genuinely fair three-way comparison. QuakeMigrate's
native .event output doesn't directly expose gap or phase count the way
NonLinLoc's .hyp files did, so these are computed here from the paired
.picks files (per-station arrival times and residuals) and our existing
station coordinates - confirmed against the real file structure before
building this, not assumed.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd

PROJECT_DIR = os.path.expanduser("~/tvz_project")
QM_DIR = os.path.join(PROJECT_DIR, "quakemigrate")
LOCATE_DIR = os.path.join(QM_DIR, "runs", "tvz_full", "locate")
CATALOG_DIR = os.path.join(PROJECT_DIR, "catalogs")

stations_df = pd.read_csv(os.path.join(QM_DIR, "lut", "stations.csv"))
stations_lookup = stations_df.set_index("Name")[["Latitude", "Longitude"]].to_dict("index")

event_files = sorted(glob.glob(os.path.join(LOCATE_DIR, "events", "*.event")))
print(f"{len(event_files)} located events found")

In [ ]:
def azimuthal_gap(event_lat, event_lon, station_names):
    azimuths = []
    for sta in station_names:
        if sta not in stations_lookup:
            continue
        sta_lat = stations_lookup[sta]["Latitude"]
        sta_lon = stations_lookup[sta]["Longitude"]
        dlon = np.radians(sta_lon - event_lon)
        lat1, lat2 = np.radians(event_lat), np.radians(sta_lat)
        x = np.sin(dlon) * np.cos(lat2)
        y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
        az = (np.degrees(np.arctan2(x, y)) + 360) % 360
        azimuths.append(az)
    if len(azimuths) < 2:
        return 360.0
    azimuths = sorted(azimuths)
    gaps = [azimuths[i+1] - azimuths[i] for i in range(len(azimuths) - 1)]
    gaps.append(360 - azimuths[-1] + azimuths[0])
    return max(gaps)


records = []
for ev_file in event_files:
    event_id = os.path.basename(ev_file).replace(".event", "")
    ev_df = pd.read_csv(ev_file)
    if len(ev_df) == 0:
        continue
    row = ev_df.iloc[0]

    picks_file = os.path.join(LOCATE_DIR, "picks", f"{event_id}.picks")
    if not os.path.exists(picks_file):
        continue
    picks_df = pd.read_csv(picks_file)
    valid_picks = picks_df[picks_df["PickTime"] != "-1"].copy()
    valid_picks["Residual"] = pd.to_numeric(valid_picks["Residual"], errors="coerce")

    n_phases = len(valid_picks)
    rms = np.sqrt((valid_picks["Residual"] ** 2).mean()) if n_phases > 0 else np.nan
    gap = azimuthal_gap(row["GAU_Y"], row["GAU_X"], valid_picks["Station"].unique())
    # COV_Err (not GAU_Err) used for horizontal error: confirmed from source
    # (quakemigrate/io/event.py) that COV_Err is explicitly computed "for use
    # when filtering", and is conceptually the same covariance-based spread
    # measure used for NonLinLoc's own error estimate (EQTransformer/PhaseNet),
    # whereas GAU_Err is a narrower Gaussian fit to the coalescence peak only -
    # a different, more optimistic quantity, not the appropriate comparison.
    horiz_error_km = np.sqrt(row["COV_ErrX"] ** 2 + row["COV_ErrY"] ** 2)

    records.append({
        "event_id": event_id,
        "origin_time": row["DT"],
        "lat": row["GAU_Y"],
        "lon": row["GAU_X"],
        "depth_km": row["GAU_Z"],
        "horiz_error_km": horiz_error_km,
        "gap": gap,
        "nphs": n_phases,
        "rms": rms,
        "ml": row["ML"],
        "ml_err": row["ML_Err"],
    })

catalog = pd.DataFrame(records)
print(f"Parsed {len(catalog)} events with paired picks data")
print(catalog[["horiz_error_km", "gap", "nphs", "rms", "ml"]].describe())

In [ ]:
# RMS threshold recalibrated for QuakeMigrate specifically (Option A,
# chosen over dropping the criterion entirely): QuakeMigrate's residual
# convention (LUT-predicted vs re-picked arrival time, averaged across all
# 60 stations by default) is not numerically comparable to NonLinLoc's own
# optimisation-based residual used for EQTransformer/PhaseNet (means of
# ~1.13s vs ~0.04-0.05s) - confirmed by the fact that even the single
# best-constrained QuakeMigrate event (0.607s) still exceeds the original
# 0.5s cutoff. 1.0s is used here instead: close to the observed median
# (1.059s), playing the same qualitative role the original threshold did
# (rejecting the more poorly-constrained half of events) without assuming
# false numerical equivalence between the two different conventions.
QM_RMS_THRESHOLD = 1.0

filtered = catalog[
    (catalog["horiz_error_km"] < 5.0) &
    (catalog["gap"] < 180.0) &
    (catalog["nphs"] >= 8) &
    (catalog["rms"] < QM_RMS_THRESHOLD)
].copy()

print(f"Events retained: {len(filtered)} of {len(catalog)} "
      f"({100 * len(filtered) / len(catalog):.1f}%)")
print("\nRejection breakdown:")
print(f"  horiz_error >= 5km: {(catalog['horiz_error_km'] >= 5.0).sum()}")
print(f"  gap >= 180deg:      {(catalog['gap'] >= 180.0).sum()}")
print(f"  nphs < 8:           {(catalog['nphs'] < 8).sum()}")
print(f"  rms >= {QM_RMS_THRESHOLD}s:        {(catalog['rms'] >= QM_RMS_THRESHOLD).sum()}")

raw_out = os.path.join(CATALOG_DIR, "quakemigrate_raw.csv")
filtered_out = os.path.join(CATALOG_DIR, "quakemigrate_quality_filtered.csv")
catalog.to_csv(raw_out, index=False)
filtered.to_csv(filtered_out, index=False)
print(f"\nSaved: {raw_out}")
print(f"Saved: {filtered_out}")